In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import pycountry
from pycountry_convert import country_name_to_country_alpha2, country_alpha2_to_continent_code
from geopy.geocoders import Nominatim

# Получаем страницу
url = "https://www.worldsdc.com/events/"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
table = soup.find("table")
data = []

# Заголовки
head = table.find("thead")
headers = [header.text for header in head.find_all("th")]
headers.extend(['canceled', 'link'])

# Считываем строки
for row in table.find_all("tr")[1:]:
    row_data = []
    for cell in row.find_all("td"):
        img = cell.find("img")
        if img and 'data-flag' in img.attrs:
            row_data.append(img['data-flag'])
        else:
            row_data.append(cell.text.strip())
    if 'event-canceled' in row.attrs.get('class', []):
        row_data.append(True)
    else:
        row_data.append(False)
    try:
        event_link = row.find("div", class_="event_name").find("a")['href']
    except:
        event_link = None
    row_data.append(event_link)
    data.append(row_data)

# Создаем DataFrame
df = pd.DataFrame(data, columns=headers)

# Парсинг даты
def format_date(date_str, year):
    try:
        return datetime.strptime(date_str + ' ' + year, '%b %d %Y').strftime('%d.%m.%Y')
    except:
        return None

def split_date(date_str):
    try:
        start_date, end_date = date_str.split(' - ')
        if ',' in end_date:
            end_date, year = end_date.split(', ')
        elif ' ' in start_date:
            start_date, year = start_date.rsplit(' ', 1)
        else:
            year = start_date.rsplit(' ', 1)[1]
        start_date = format_date(start_date, year)
        end_date = format_date(end_date, year)
        return pd.Series([year, start_date, end_date])
    except:
        return pd.Series([None, None, None])

df[['year', 'start_date', 'end_date']] = df['Date'].apply(split_date)
df.drop(columns=['end_date'], inplace=True)

# Обработка названий событий
def split_event_name(event_name):
    status_event = ''
    if 'Registry Event' in event_name:
        status_event = 'Registry Event'
        event_name = event_name.replace('Registry Event', '')
    elif 'Trial Event' in event_name:
        status_event = 'Trial Event'
        event_name = event_name.replace('Trial Event', '').replace('(Trial Event)', '')
    confirmed = not any(x in event_name.lower() for x in ['unconfirmed', 'unconirmed', 'unfonfirmed'])
    for word in ['(Unconfirmed)', '(unconfirmed)', '(Unfonfirmed)', 'Unconfirmed', 'unconirmed']:
        event_name = event_name.replace(word, '')
    return pd.Series([event_name.strip(), confirmed, status_event])

df[['Event Name', 'confirmed', 'status_event']] = df['Event Name'].apply(split_event_name)

# Обработка стран
df.loc[df['Event Location'] == 'Windsor/Slough, London', 'Country'] = 'GBR'
df['Country'] = df['Country'].replace('', 'RUS')

def convert_country_code_to_name(code):
    try:
        return pycountry.countries.get(alpha_3=code).name
    except:
        return None

df['Country'] = df['Country'].apply(convert_country_code_to_name)

df.rename(columns={
    'Date': 'original_date',
    'Event Name': 'event_name',
    'Event Location': 'event_location',
    'Country': 'country'
}, inplace=True)

df = df.reindex(columns=[
    'year', 'start_date', 'original_date', 'event_name',
    'status_event', 'event_location', 'country', 'confirmed',
    'canceled', 'link'
])

# Поправка вручную стран/локаций
manual_fixes = {
    'Ljubljana, Slovenia': 'Slovenia',
    'Zurich,  Swintzerland': 'Switzerland',
    'Kraków, malopolska, Polska': 'Poland',
    'LYON France, Rhones, France': 'France',
    'Bonn, Nordrhein-Westphalen, Germany': 'Germany',
    'LYON, Rhones, France': 'France',
    'Slovenia': 'Slovenia',
}
for location, country in manual_fixes.items():
    df.loc[df['event_location'] == location, 'country'] = country

# Континенты
def create_continent(country):
    try:
        if country:
            code = country_name_to_country_alpha2(country, cn_name_format="default")
            return country_alpha2_to_continent_code(code)
        return None
    except:
        return None

df['continent'] = df['country'].apply(create_continent)

# Геолокация
geolocator = Nominatim(user_agent="my_geocoder")
cache = {}

def get_coordinates(location):
    if location in cache:
        return cache[location]
    try:
        loc = geolocator.geocode(location, timeout=10)
        if loc:
            coords = (loc.latitude, loc.longitude)
            cache[location] = coords
            return coords
        else:
            return None
    except Exception as e:
        print(f"Geocode error for {location}: {e}")
        return None

df["Coordinates"] = df["event_location"].apply(get_coordinates)
df[["Latitude", "Longitude"]] = df["Coordinates"].apply(pd.Series)
df.drop(columns=["Coordinates"], inplace=True)

# Ручные координаты
additional_coords = {
    "Sesto San Giovani, Italy": (45.5328, 9.2257),
    "Old Windsor, Berkshire, Engand": (51.46225, -0.58059),
    "Bracknell, Berkshire, United Kingdom": (51.416, -0.749),
    "Westminster, Col": (39.83665, -105.0372),
    "Mansfield, MA": (42.0333, -71.2194),
    "Helsinki, Finland": (60.1695, 24.9354),
    "Philadelphia PA, Delaware, New Castle": (39.5783, -75.639),
    "San Francisco, CA, San Mateo": (37.5542, -122.313),
    "Seatttle, WA, King": (47.608, -122.335),
    "Stockholm-Uppsala, Uppland, Uppland": (59.9479, 18.0621),
    "Singapore": (1.2897, 103.85),
    "Auckland, North Island, New Zealand": (-36.8485, 174.763)
}

for loc, coords in additional_coords.items():
    mask = df['event_location'] == loc
    df.loc[mask, ['Latitude', 'Longitude']] = coords

# Сохраняем результат
df.to_csv("event_list_for_calendar.csv", index=False, encoding='utf-8-sig')
